# Staying Power in Top Novels and NYT Bestsellers

This notebook explores two datasets. The Goodreads Top 500 Novels and the NYT Bestseller list (1931-2020). Both lists are ways of saying "this book mattered," but they measure different things. Goodreads top ranked novels are books that lots of readers rated highly long after publication. NYT bestsellers are books that sold well in a given week. The question I want to look at is which books have staying power by which measure and how much overlap there really is.

## Setup

In [8]:
import pandas as pd
import altair as alt

#alt.data_transformers.disable_max_rows()

novels_df = pd.read_csv(
    "https://raw.githubusercontent.com/melaniewalsh/responsible-datasets-in-context/main/datasets/top-500-novels/final_merged_dataset_no_full_text.tsv",
    sep="\t"
)

nyt_df = pd.read_csv(
    "https://raw.githubusercontent.com/ecds/post45-datasets/main/nyt_full.tsv",
    sep="\t"
)

print(novels_df.shape, nyt_df.shape)

(500, 29) (60386, 6)


## Cleaning

The NYT titles are in all caps and the novels titles aren't, so I lowercase both for matching. The Goodreads ratings columns have commas in them 269,435 so I strip those and convert to numeric.

In [9]:
novels_df["title_lower"] = novels_df["title"].str.lower().str.strip()
novels_df["author_lower"] = novels_df["author"].str.lower().str.strip()
nyt_df["title_lower"] = nyt_df["title"].str.lower().str.strip()
nyt_df["author_lower"] = nyt_df["author"].str.lower().str.strip()

novels_df["gr_num_ratings"] = novels_df["gr_num_ratings"].str.replace(",", "").astype(float)
novels_df["gr_avg_rating"] = pd.to_numeric(novels_df["gr_avg_rating"], errors="coerce")

## When were the top novels published?

In [10]:
alt.Chart(novels_df).mark_bar().encode(
    x=alt.X("pub_year:Q", bin=alt.Bin(step=20), title="Publication Year"),
    y=alt.Y("count():Q", title="Number of Novels")
).properties(title="When the Top 500 Novels Were Published", width=500, height=250)

alt.Chart(...)

Most of the top novels come from the 1800s and 1900s, peaking around the early 20th century. There's a clear drop off after about 1980. This gives an idea that in order to have staying power books need a lot of time, decades even, to get that long standing attention that lands them on a top ranked list. A book published in 2015 lacks the time and "classiness" of these older books. 

## How long do bestsellers last on the NYT list?

Each NYT row is a single weekly listing, so I'll group by title to count how many weeks each book spent on the list.

In [11]:
nyt_summary = (
    nyt_df.groupby(["title_lower", "author_lower"])
    .agg(title=("title", "first"), author=("author", "first"), weeks_on_list=("week", "count"))
    .reset_index()
)

alt.Chart(nyt_summary).mark_bar().encode(
    x=alt.X("weeks_on_list:Q", bin=alt.Bin(step=5), title="Weeks on Bestseller List"),
    y=alt.Y("count():Q", title="Number of Books")
).properties(title="How Long Books Stay on the NYT Bestseller List", width=500, height=250)

alt.Chart(...)

The vast majority of bestsellers spend just a handful of weeks on the list before falling off. Only a small group of books make it past 30 weeks, and the long-runners (a year or more) are rare. So when you sort bestsellers by total weeks on the list, you're surfacing books that genuinely had cultural staying power in their moment.

## Do long-running bestsellers also make the top novels list?

In [12]:
top500_keys = set(zip(novels_df["title_lower"], novels_df["author_lower"]))
nyt_summary["in_top500"] = [
    (t, a) in top500_keys for t, a in zip(nyt_summary["title_lower"], nyt_summary["author_lower"])
]

top20 = nyt_summary.nlargest(20, "weeks_on_list")

alt.Chart(top20).mark_bar().encode(
    x=alt.X("weeks_on_list:Q", title="Weeks on Bestseller List"),
    y=alt.Y("title:N", sort="-x"),
    color=alt.Color("in_top500:N", title="In Top 500?")
).properties(title="Top 20 Longest-Running Bestsellers", width=450, height=400)

alt.Chart(...)

Most of the longest running bestsellers don't end up on the Goodreads top novels list. The books that survived on the NYT list for a year or more were usually thrillers, romances, and other popular fiction. They sold incredibly well in their moment but mostly didn't transition into long term reader recognitions. Selling well for a long time and being highly rated decades later turn out to be pretty different things.

## Does an author's NYT presence relate to their position in the top novels list?

I'll tag each top novel with whether its author ever appeared on the NYT list, then see if there's a difference in average Goodreads rating between the two groups.

In [13]:
nyt_authors = set(nyt_df["author_lower"].unique())
novels_df["author_charted"] = novels_df["author_lower"].isin(nyt_authors)

alt.Chart(novels_df.dropna(subset=["gr_avg_rating"])).mark_boxplot().encode(
    x=alt.X("author_charted:N", title="Author appeared on NYT?"),
    y=alt.Y("gr_avg_rating:Q", title="Goodreads Average Rating", scale=alt.Scale(zero=False))
).properties(title="Goodreads Ratings: Top 500 Authors With and Without NYT Appearances", width=400, height=300)

alt.Chart(...)

Authors whose books also charted on NYT actually have slightly higher median Goodreads ratings than those who never did. This pushes back a little on the previous chart. At the author level, commercial success and reader rated quality aren't completely disconnected. But the overlap is happening more at the author level than the individual book level. For example, a bestselling author might have one book on each list, not the same book.

## Genre patterns in the top novels

In [14]:
alt.Chart(novels_df[novels_df["genre"] != "na"]).mark_bar().encode(
    x=alt.X("count():Q", title="Number of Novels"),
    y=alt.Y("genre:N", sort="-x")
).properties(title="Genres in the Top 500", width=400, height=300)

alt.Chart(...)

Action and fantasy are surprisingly well represented for a list of top ranked literary novels. The genre column has some quirks (entries like **bildung** and **picaresque** are more era tags than genres) but the bigger picture is that the top novels list isn't strictly literary fiction. Things like adventure and fantasy all show up. This matters for the comparison with NYT because it shows that the top ranked novels list and the bestseller list aren't actually drawing from totally different genre pools.

## Gaps and biases

A few things worth keeping in mind about what these datasets can and can't tell us:

The Top 500 is a curated list, not an objective measure. A different curator with the same goal would produce a different 500, especially around the edges.

The NYT list is American. International bestsellers and books that sold mostly through book clubs or schools wouldn't show up.

I matched authors and titles by lowercased exact strings, which means "Pearl S. Buck" and "Pearl Buck" count as different people. That could lead to gaps in my data analysis but thats what makes checking books so hard.

The two datasets cover overlapping but different time spans. NYT starts in 1931 while the top novels list goes back centuries, so any author from before 1931 is automatically excluded from the NYT side regardless of how their books would have sold.